In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tools import get_current_time
import numpy as np

/tmp/ipykernel_28560/3874352872.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/nineleaps/Documents/Ai agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader=PyPDFLoader("data/DBMS.pdf")
docs=loader.load()
print("PDF Loaded")

PDF Loaded


In [3]:
sp=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
ds=sp.split_documents(docs)
print("Chunks Created:",len(ds))

Chunks Created: 813


In [4]:
texts=[d.page_content for d in ds]
vec=TfidfVectorizer()
vectors=vec.fit_transform(texts)
print("Embeddings Created")

Embeddings Created


In [5]:
chat_history = []

In [6]:
def rewrite_question(question):
    if len(chat_history)==0:
        return question
    if "it" in question.lower():
        previous_question=chat_history[-1]["question"]
        return question.replace(
            "it",
            previous_question
        )
    return question

In [7]:
def retrieve_context(question):
    query_vector=vec.transform([question])
    similarities=cosine_similarity(
        query_vector,
        vectors
    )[0]
    best_index=np.argmax(similarities)
    best_score=similarities[best_index]
    context=texts[best_index]
    return context,best_score

In [8]:
def call_tool(question,score):
    if "time" in question.lower():
        return True
    if score < 0.10:
        return True
    return False

In [10]:
question=input("Enter question: ")
question1=rewrite_question(question)
context,score=retrieve_context(question1)
if call_tool(question1,score):
    result=get_current_time()
    answer=f"Current Time: {result}"
    print("\nTool Called")
    print(answer)
else:
    answer=context
    print("\nAnswer:")
    print(answer)
chat_history.append({"question":question,"answer":answer})


Answer:
requiring that each non-key attribute be dependent on the primary key. 
This means that each column should be directly related to the primary 
key, and not to other columns.
